[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AImotion-Bavaria/kipro_wi/blob/main/lecture2/australia_ortools.ipynb)

# Map Colouring Australia with OR-Tools

Every region of Australia gets one of three colours, so that
**neighbouring regions never have the same colour**.

In [ ]:
!pip install -q ortools

## Setup

The drawing code lives in `australia_viz.py`. On Colab it is downloaded from the
GitHub repository (locally it is found next to the notebook).

When running locally, `autoreload` re-imports the module automatically, so changes to
`australia_viz.py` take effect without restarting the kernel. On Colab it is switched off.

In [ ]:
import os
import urllib.request

if not os.path.exists("australia_viz.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/AImotion-Bavaria/kipro_wi/main/lecture2/australia_viz.py",
        "australia_viz.py")

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules

# autoreload only for local development (not needed on Colab)
if not IN_COLAB:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")

import matplotlib.pyplot as plt
import australia_viz as viz

## Model

CP-SAT has no enum type like MiniZinc, so each colour is a number:
`0 = blue`, `1 = green`, `2 = red`.

In [ ]:
from ortools.sat.python import cp_model

# possible values
colors = ["blue", "green", "red"]
regions = ["WA", "NT", "SA", "QL", "NSW", "VT"]

# pairs of regions that share a border
borders = [("WA", "NT"), ("SA", "NT"), ("WA", "SA"), ("QL", "NT"), ("QL", "SA"),
           ("NSW", "SA"), ("NSW", "VT"), ("NSW", "QL"), ("VT", "SA")]

model = cp_model.CpModel()

# decision variables: one colour index per region
x = {r: model.new_int_var(0, len(colors) - 1, r) for r in regions}

# constraints: bordering regions must be different
for a, b in borders:
    model.add(x[a] != x[b])

## Solve

In [ ]:
solver = cp_model.CpSolver()
status = solver.solve(model)

if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    solution = {r: colors[solver.value(x[r])] for r in regions}
    for r, c in solution.items():
        print(f"{r} = {c};")
else:
    print("=====UNSATISFIABLE=====")

## Visualisation

A simplified map of Australia: regions touch exactly where the model has a border constraint.
The drawing functions come from `australia_viz.py`.

In [ ]:
viz.draw_map(solution)
plt.show()

## All solutions

CP-SAT can also enumerate **every** valid colouring.
A solution callback collects each solution the solver finds.

In [ ]:
class CollectSolutions(cp_model.CpSolverSolutionCallback):
    def __init__(self, variables):
        super().__init__()
        self.variables = variables
        self.solutions = []

    def on_solution_callback(self):
        self.solutions.append({r: colors[self.value(v)] for r, v in self.variables.items()})


collector = CollectSolutions(x)
solver = cp_model.CpSolver()
solver.parameters.enumerate_all_solutions = True
solver.solve(model, collector)

print(f"{len(collector.solutions)} solutions found")

In [ ]:
viz.draw_all(collector.solutions)
plt.show()